In [1]:
import pandas as pd
vocab_df = pd.read_csv('unique_corpus_sentences/urdu_vocab.tsv',sep = '\t')
vocab_set = set()

In [2]:
URDU_0_99 = {
    0: "صفر", 1: "ایک", 2: "دو", 3: "تین", 4: "چار", 5: "پانچ", 6: "چھ", 7: "سات", 8: "آٹھ", 9: "نو",
    10: "دس", 11: "گیارہ", 12: "بارہ", 13: "تیرہ", 14: "چودہ", 15: "پندرہ", 16: "سولہ", 17: "سترہ", 18: "اٹھارہ", 19: "انیس",
    20: "بیس", 21: "اکیس", 22: "بائیس", 23: "تئیس", 24: "چوبیس", 25: "پچیس", 26: "چھبیس", 27: "ستائیس", 28: "اٹھائیس", 29: "انتیس",
    30: "تیس", 31: "اکتیس", 32: "بتیس", 33: "تینتیس", 34: "چونتیس", 35: "پینتیس", 36: "چھتیس", 37: "سینتیس", 38: "اڑتیس", 39: "انتالیس",
    40: "چالیس", 41: "اکتالیس", 42: "بیالیس", 43: "تینتالیس", 44: "چوالیس", 45: "پینتالیس", 46: "چھیالیس", 47: "سینتالیس", 48: "اڑتالیس", 49: "انچاس",
    50: "پچاس", 51: "اکیاون", 52: "باون", 53: "ترپن", 54: "چون", 55: "پچپن", 56: "چھپن", 57: "ستاون", 58: "اٹھاون", 59: "انسٹھ",
    60: "ساٹھ", 61: "اکسٹھ", 62: "باسٹھ", 63: "ترسٹھ", 64: "چونسٹھ", 65: "پینسٹھ", 66: "چھیاسٹھ", 67: "سڑسٹھ", 68: "اڑسٹھ", 69: "انہتر",
    70: "ستر", 71: "اکہتر", 72: "بہتر", 73: "تہتر", 74: "چوہتر", 75: "پچہتر", 76: "چھہتر", 77: "ستتر", 78: "اٹھہتر", 79: "اناسی",
    80: "اسی", 81: "اکیاسی", 82: "بیاسی", 83: "تراسی", 84: "چوراسی", 85: "پچاسی", 86: "چھیاسی", 87: "ستاسی", 88: "اٹھاسی", 89: "نواسی",
    90: "نوے", 91: "اکانوے", 92: "بانوے", 93: "ترانوے", 94: "چورانوے", 95: "پچانوے", 96: "چھیانوے", 97: "ستانوے", 98: "اٹھانوے", 99: "ننانوے"
}

def num_to_urdu(n):
    """Convert integer to Urdu words (space-separated, CER-friendly)"""
    
    parts = []

    if n >= 10000000:  # crore
        crore = n // 10000000
        n %= 10000000
        parts.append("ایک کروڑ" if crore == 1 else num_to_urdu(crore) + " کروڑ")

    if n >= 100000:  # lakh
        lakh = n // 100000
        n %= 100000
        parts.append("ایک لاکھ" if lakh == 1 else num_to_urdu(lakh) + " لاکھ")

    if n >= 1000:  # thousand
        thousand = n // 1000
        n %= 1000
        parts.append("ایک ہزار" if thousand == 1 else num_to_urdu(thousand) + " ہزار")

    if n >= 100:  # hundred
        hundred = n // 100
        n %= 100
        parts.append("ایک سو" if hundred == 1 else URDU_0_99[hundred] + " سو")

    if n > 0:
        parts.append(URDU_0_99[n])
    elif not parts:  # n == 0
        parts.append(URDU_0_99[0])

    return " ".join(parts)

In [3]:
import pandas as pd
import re

def normalize_urdu(text):
    """
    Ultra-aggressive Urdu normalizer - nukes everything except core Urdu chars
    """
    if pd.isna(text) or text == '':
        return ''
    
    # 1. Remove diacritics (zabar, zer, pesh, tanwin, shadda, sukun, etc.)
    text = re.sub(r'[\u0610-\u061A\u064B-\u065F\u0670]', '', text)
    
    # 2. Remove tatweel/kashida (ـ)
    text = text.replace('\u0640', '')
    
    # 3. Normalize Arabic → Urdu
    char_map = {
        '\u0643': '\u06A9',  # Arabic KAF → Urdu KEHEH (ك → ک)
        '\u0647': '\u06C1',  # Arabic HEH → Urdu HEH GOAL (ه → ہ)
        '\u0649': '\u06CC',  # ALEF MAQSURA → Farsi YEH (ى → ی)
        '\u0629': '\u06C1',  # TEH MARBUTA → Urdu HEH GOAL (ة → ہ)
        '\u0623': '\u0627',  # ALEF+HAMZA ABOVE → ALEF (أ → ا)
        '\u0625': '\u0627',  # ALEF+HAMZA BELOW → ALEF (إ → ا)
        '\u064A': '\u06CC',  # Arabic YEH → Farsi YEH (ي → ی)
    }
    for src, tgt in char_map.items():
        text = text.replace(src, tgt)
    
    # 4. Remove zero-width & invisible chars
    text = re.sub(r'[\u200B-\u200F\u202A-\u202E\u2060-\u206F\uFEFF]', '', text)
    text = re.sub(r'[\u2018-\u201F\u0060\u00B4\u02BC]', '', text)
    
    # 5. NUKE ALL PUNCTUATION (Urdu + ASCII + special)
    # Urdu punctuation block
    text = re.sub(r'[،؍؎؏۔؟؛٪٬]', '', text)
    # Extended Arabic punctuation
    text = re.sub(r'[\u0600-\u060C\u060E-\u061F]', '', text)
    # ASCII punctuation & symbols (EVERYTHING)
    text = re.sub(r'[!@#$%^&*()_+=\[\]{};:\'",.<>/?\\|`~\-–—''""…]', '', text)
    
    # 6. CONVERT DIGITS TO URDU WORDS, then remove any remaining English letters
    text = re.sub(r'[0-9]+', lambda m: num_to_urdu(int(m.group())), text)
    text = re.sub(r'[A-Za-z]', '', text)
    
    # 7. Collapse all whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # 8. Keep only Urdu characters and whitespace
    text = re.sub(r'[^\u0600-\u06FF\u0750-\u077F\s]', '', text)
    return text

In [4]:
import re
import pickle
from collections import defaultdict, Counter
from pathlib import Path

BIGRAM_CHECKPOINT_DIR = Path("checkpoints/bigrams")
TRIGRAM_CHECKPOINT_DIR = Path("checkpoints/trigrams")

for dir_path in [BIGRAM_CHECKPOINT_DIR, TRIGRAM_CHECKPOINT_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

BIGRAM_THRESHOLD = 1000_000
TRIGRAM_THRESHOLD = 1000_000

TOTAL_EST = 10_000_000
MILESTONE = 500_000

bigram_forward = defaultdict(Counter)
bigram_backward = defaultdict(Counter)
trigram_left = defaultdict(Counter)
trigram_middle = defaultdict(Counter)
trigram_right = defaultdict(Counter)

dump_counter = 0

def dump_to_disk(data_dict, name, dump_id):
    filepath = None
    if "bigram" in name:
        filepath = BIGRAM_CHECKPOINT_DIR / f"{name}_{dump_id:04d}.pkl"
    elif "trigram" in name:
        filepath = TRIGRAM_CHECKPOINT_DIR / f"{name}_{dump_id:04d}.pkl"
    
    with open(filepath, 'wb') as f:
        pickle.dump(dict(data_dict), f)
    
    print(f"Dumped {name}: {len(data_dict):,} entries → {filepath.name}")
    return defaultdict(Counter) if isinstance(data_dict, defaultdict) else Counter()

def should_dump_bigrams():
    return (len(bigram_forward) > BIGRAM_THRESHOLD or 
            len(bigram_backward) > BIGRAM_THRESHOLD)

def should_dump_trigrams():
    return (len(trigram_left) > TRIGRAM_THRESHOLD or 
            len(trigram_middle) > TRIGRAM_THRESHOLD or 
            len(trigram_right) > TRIGRAM_THRESHOLD)

print(f"Vocabulary size: {len(vocab_set):,} words")
print("Processing corpus...\n")

with open('ur.txt', 'r', encoding='utf-8') as f:
    for line_num, line in enumerate(f, 1):
        norm = normalize_urdu(line)
        
        tokens = norm.split()
        n = len(tokens)
        if n < 1:
            continue

        for token_idx in range(n - 1):
            w0, w1 = tokens[token_idx], tokens[token_idx + 1]
            bigram_forward[w0][w1] += 1
            bigram_backward[w1][w0] += 1

        for token_idx in range(n - 2):
            w0, w1, w2 = tokens[token_idx], tokens[token_idx + 1], tokens[token_idx + 2]
            trigram_left[(w1, w2)][w0] += 1
            trigram_middle[(w0, w2)][w1] += 1
            trigram_right[(w0, w1)][w2] += 1

        if should_dump_bigrams():
            bigram_forward = dump_to_disk(bigram_forward, "bigram_forward", dump_counter)
            bigram_backward = dump_to_disk(bigram_backward, "bigram_backward", dump_counter)
            dump_counter += 1

        if should_dump_trigrams():
            trigram_left = dump_to_disk(trigram_left, "trigram_left", dump_counter)
            trigram_middle = dump_to_disk(trigram_middle, "trigram_middle", dump_counter)
            trigram_right = dump_to_disk(trigram_right, "trigram_right", dump_counter)
            dump_counter += 1

        if line_num % MILESTONE == 0:
            pct = min(line_num / TOTAL_EST * 100, 100)
            mem_info = f"BF:{len(bigram_forward):,} BB:{len(bigram_backward):,} TL:{len(trigram_left):,}"
            print(f"  [{pct:5.1f}%] {line_num:>10,} lines | {mem_info}")

if bigram_forward or bigram_backward:
    print("\nFinal bigram dump...")
    bigram_forward = dump_to_disk(bigram_forward, "bigram_forward", dump_counter)
    bigram_backward = dump_to_disk(bigram_backward, "bigram_backward", dump_counter)
    dump_counter += 1

if trigram_left or trigram_middle or trigram_right:
    print("Final trigram dump...")
    trigram_left = dump_to_disk(trigram_left, "trigram_left", dump_counter)
    trigram_middle = dump_to_disk(trigram_middle, "trigram_middle", dump_counter)
    trigram_right = dump_to_disk(trigram_right, "trigram_right", dump_counter)

Vocabulary size: 0 words
Processing corpus...

Dumped trigram_left: 688,848 entries → trigram_left_0000.pkl
Dumped trigram_middle: 1,000,009 entries → trigram_middle_0000.pkl
Dumped trigram_right: 694,390 entries → trigram_right_0000.pkl
Dumped trigram_left: 690,949 entries → trigram_left_0001.pkl
Dumped trigram_middle: 1,000,002 entries → trigram_middle_0001.pkl
Dumped trigram_right: 697,110 entries → trigram_right_0001.pkl
Dumped trigram_left: 691,446 entries → trigram_left_0002.pkl
Dumped trigram_middle: 1,000,020 entries → trigram_middle_0002.pkl
Dumped trigram_right: 697,316 entries → trigram_right_0002.pkl
Dumped trigram_left: 693,499 entries → trigram_left_0003.pkl
Dumped trigram_middle: 1,000,013 entries → trigram_middle_0003.pkl
Dumped trigram_right: 699,931 entries → trigram_right_0003.pkl
Dumped trigram_left: 690,830 entries → trigram_left_0004.pkl
Dumped trigram_middle: 1,000,036 entries → trigram_middle_0004.pkl
Dumped trigram_right: 696,342 entries → trigram_right_0004.pk

In [5]:
import pickle
import pandas as pd
import duckdb
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

SRC = Path("checkpoints/trigrams")
DST = Path("checkpoints/parquet")
MERGED = Path("checkpoints/merged")

for p in [DST, MERGED]:
    p.mkdir(parents=True, exist_ok=True)

def pkl_to_parquet(pkl_path):
    pkl_path = Path(pkl_path)
    with open(pkl_path, 'rb') as f:
        data = pickle.load(f)
    rows = [(k[0], k[1], v, cnt)
            for k, counter in data.items()
            for v, cnt in counter.items()]
    if not rows:
        return
    df = pd.DataFrame(rows, columns=['k0','k1','v','cnt'])
    out = DST / pkl_path.name.replace('.pkl', '.parquet')
    df.to_parquet(out, index=False)

print("Step 1: Converting pkl → parquet...\n")
for name in ["trigram_left", "trigram_middle", "trigram_right"]:
    chunks = [str(c) for c in sorted(SRC.glob(f"{name}_*.pkl"))]
    print(f"  {name}: {len(chunks)} files")
    with ThreadPoolExecutor(8) as pool:
        list(tqdm(pool.map(pkl_to_parquet, chunks), total=len(chunks),
                desc=f"  {name}", unit="file"))

print("Step 2: DuckDB merge...\n")

for name in ["trigram_left", "trigram_middle", "trigram_right"]:
    out = MERGED / f"{name}.parquet"
    print(f"  merging {name}...")
    duckdb.sql(f"""
        COPY (
            SELECT k0, k1, v, SUM(cnt) as cnt
            FROM read_parquet('{DST}/{name}_*.parquet')
            GROUP BY k0, k1, v
        ) TO '{out}' (FORMAT PARQUET)
    """)
    rows = duckdb.sql(f"SELECT COUNT(*) FROM read_parquet('{out}')").fetchone()[0]
    print(f"  {name} ✅ → {rows:,} rows\n")

print("All done.")

Step 1: Converting pkl → parquet...

  trigram_left: 268 files


  trigram_left: 100%|██████████| 268/268 [36:40<00:00,  8.21s/file] 


  trigram_middle: 268 files


  trigram_middle: 100%|██████████| 268/268 [43:56<00:00,  9.84s/file] 


  trigram_right: 268 files


  trigram_right: 100%|██████████| 268/268 [36:47<00:00,  8.24s/file]


Step 2: DuckDB merge...

  merging trigram_left...
  trigram_left ✅ → 126,130,890 rows

  merging trigram_middle...
  trigram_middle ✅ → 126,130,890 rows

  merging trigram_right...
  trigram_right ✅ → 126,130,890 rows

All done.


In [6]:
def pkl_to_parquet_bigram(pkl_path):
    pkl_path = Path(pkl_path)
    with open(pkl_path, 'rb') as f:
        data = pickle.load(f)
    rows = [(k, v, cnt)
            for k, counter in data.items()
            for v, cnt in counter.items()]
    if not rows:
        return
    df = pd.DataFrame(rows, columns=['k0','v','cnt'])
    out = DST / pkl_path.name.replace('.pkl', '.parquet')
    df.to_parquet(out, index=False)

In [8]:
print("Step 1: Converting pkl → parquet...\n")
for name in ["bigram_forward", "bigram_backward"]:
    chunks = [str(c) for c in sorted(Path('checkpoints/bigrams').glob(f"{name}_*.pkl"))]
    print(f"  {name}: {len(chunks)} files")
    with ThreadPoolExecutor(8) as pool:
        list(tqdm(pool.map(pkl_to_parquet_bigram, chunks), total=len(chunks),
                desc=f"  {name}", unit="file"))

print("Step 2: DuckDB merge...\n")

for name in ["bigram_forward", "bigram_backward"]:
    out = MERGED / f"{name}.parquet"
    print(f"  merging {name}...")
    duckdb.sql(f"""
        COPY (
            SELECT k0, v, SUM(cnt) as cnt
            FROM read_parquet('{DST}/{name}_*.parquet')
            GROUP BY k0, v
        ) TO '{out}' (FORMAT PARQUET)
    """)
    rows = duckdb.sql(f"SELECT COUNT(*) FROM read_parquet('{out}')").fetchone()[0]
    print(f"  {name} ✅ → {rows:,} rows\n")

print("All done.")

Step 1: Converting pkl → parquet...

  bigram_forward: 5 files


  bigram_forward: 100%|██████████| 5/5 [03:18<00:00, 39.73s/file] 


  bigram_backward: 5 files


  bigram_backward: 100%|██████████| 5/5 [02:47<00:00, 33.59s/file] 


Step 2: DuckDB merge...

  merging bigram_forward...
  bigram_forward ✅ → 31,204,242 rows

  merging bigram_backward...
  bigram_backward ✅ → 31,204,242 rows

All done.


In [10]:
import duckdb
from pathlib import Path

MERGED = Path("checkpoints/merged")  # adjust to your actual MERGED path

def check_bigrams():
    for name in ["bigram_forward", "bigram_backward"]:
        out = MERGED / f"{name}.parquet"
        
        if not out.exists():
            print(f"❌ {name}.parquet not found at {out}")
            continue

        print(f"\n{'='*50}")
        print(f"📦 {name}")
        print(f"{'='*50}")

        # Basic stats
        total = duckdb.sql(f"SELECT COUNT(*) FROM read_parquet('{out}')").fetchone()[0]
        total_cnt = duckdb.sql(f"SELECT SUM(cnt) FROM read_parquet('{out}')").fetchone()[0]
        unique_k0 = duckdb.sql(f"SELECT COUNT(DISTINCT k0) FROM read_parquet('{out}')").fetchone()[0]

        print(f"  Total rows     : {total:,}")
        print(f"  Total cnt sum  : {total_cnt:,}")
        print(f"  Unique k0s     : {unique_k0:,}")

        # Schema
        print(f"\n  Schema:")
        schema = duckdb.sql(f"DESCRIBE SELECT * FROM read_parquet('{out}') LIMIT 1").fetchall()
        for col in schema:
            print(f"    {col[0]:<10} {col[1]}")

        # Sample rows
        print(f"\n  Top 10 by cnt:")
        rows = duckdb.sql(f"""
            SELECT k0, v, cnt
            FROM read_parquet('{out}')
            ORDER BY cnt DESC
            LIMIT 10
        """).fetchall()
        print(f"  {'k0':<20} {'v':<20} {'cnt':>10}")
        print(f"  {'-'*20} {'-'*20} {'-'*10}")
        for k0, v, cnt in rows:
            print(f"  {str(k0):<20} {str(v):<20} {cnt:>10,}")

        # Distribution: how many (k0, v) pairs per k0
        print(f"\n  k0 fan-out (avg v's per k0): "
              f"{total / unique_k0:.1f}")

        # Nulls check
        nulls_k0 = duckdb.sql(f"SELECT COUNT(*) FROM read_parquet('{out}') WHERE k0 IS NULL").fetchone()[0]
        nulls_v  = duckdb.sql(f"SELECT COUNT(*) FROM read_parquet('{out}') WHERE v IS NULL").fetchone()[0]
        nulls_cnt= duckdb.sql(f"SELECT COUNT(*) FROM read_parquet('{out}') WHERE cnt IS NULL").fetchone()[0]
        print(f"  Nulls → k0: {nulls_k0}, v: {nulls_v}, cnt: {nulls_cnt}")

check_bigrams()


📦 bigram_forward
  Total rows     : 31,204,242
  Total cnt sum  : 700,795,986.0
  Unique k0s     : 2,760,282

  Schema:
    k0         VARCHAR
    v          VARCHAR
    cnt        DOUBLE

  Top 10 by cnt:
  k0                   v                           cnt
  -------------------- -------------------- ----------
  ہے                   کہ                   2,467,947.0
  کے                   لیے                  1,689,437.0
  کے                   لئے                  1,331,339.0
  کے                   ساتھ                 1,330,678.0
  کے                   بعد                  1,197,187.0
  ہے                   اور                  1,136,976.0
  اس                   کے                   1,008,439.0
  دو                   ہزار                  959,761.0
  نے                   کہا                   899,106.0
  کے                   مطابق                 869,018.0

  k0 fan-out (avg v's per k0): 11.3
  Nulls → k0: 0, v: 0, cnt: 0

📦 bigram_backward
  Total rows     : 31,204,242
  Total cn

In [11]:
import duckdb
from pathlib import Path

MERGED = Path("checkpoints/merged")

def check_trigrams():
    for name in ["trigram_left", "trigram_middle", "trigram_right"]:
        out = MERGED / f"{name}.parquet"

        if not out.exists():
            print(f"❌ {name}.parquet not found at {out}")
            continue

        print(f"\n{'='*50}")
        print(f"📦 {name}")
        print(f"{'='*50}")

        # Basic stats
        total     = duckdb.sql(f"SELECT COUNT(*) FROM read_parquet('{out}')").fetchone()[0]
        total_cnt = duckdb.sql(f"SELECT SUM(cnt) FROM read_parquet('{out}')").fetchone()[0]
        unique_k0 = duckdb.sql(f"SELECT COUNT(DISTINCT k0) FROM read_parquet('{out}')").fetchone()[0]
        unique_k1 = duckdb.sql(f"SELECT COUNT(DISTINCT k1) FROM read_parquet('{out}')").fetchone()[0]
        unique_pairs = duckdb.sql(f"SELECT COUNT(DISTINCT (k0, k1)) FROM read_parquet('{out}')").fetchone()[0]

        print(f"  Total rows        : {total:,}")
        print(f"  Total cnt sum     : {total_cnt:,}")
        print(f"  Unique k0s        : {unique_k0:,}")
        print(f"  Unique k1s        : {unique_k1:,}")
        print(f"  Unique (k0,k1)    : {unique_pairs:,}")

        # Schema
        print(f"\n  Schema:")
        schema = duckdb.sql(f"DESCRIBE SELECT * FROM read_parquet('{out}') LIMIT 1").fetchall()
        for col in schema:
            print(f"    {col[0]:<10} {col[1]}")

        # Top 10 by cnt
        print(f"\n  Top 10 by cnt:")
        rows = duckdb.sql(f"""
            SELECT k0, k1, v, cnt
            FROM read_parquet('{out}')
            ORDER BY cnt DESC
            LIMIT 10
        """).fetchall()
        print(f"  {'k0':<20} {'k1':<20} {'v':<20} {'cnt':>10}")
        print(f"  {'-'*20} {'-'*20} {'-'*20} {'-'*10}")
        for k0, k1, v, cnt in rows:
            print(f"  {str(k0):<20} {str(k1):<20} {str(v):<20} {cnt:>10,.0f}")

        # Fan-out: avg v's per (k0, k1) pair
        print(f"\n  (k0,k1) fan-out (avg v's per pair): {total / unique_pairs:.1f}")

        # Null check
        nulls = {
            col: duckdb.sql(f"SELECT COUNT(*) FROM read_parquet('{out}') WHERE {col} IS NULL").fetchone()[0]
            for col in ["k0", "k1", "v", "cnt"]
        }
        print(f"  Nulls → " + ", ".join(f"{c}: {n}" for c, n in nulls.items()))

check_trigrams()


📦 trigram_left
  Total rows        : 126,130,890
  Total cnt sum     : 675,111,599.0
  Unique k0s        : 2,623,497
  Unique k1s        : 2,626,586
  Unique (k0,k1)    : 30,343,744

  Schema:
    k0         VARCHAR
    k1         VARCHAR
    v          VARCHAR
    cnt        DOUBLE

  Top 10 by cnt:
  k0                   k1                   v                           cnt
  -------------------- -------------------- -------------------- ----------
  بارے                 میں                  کے                      573,136
  کہا                  کہ                   نے                      546,896
  جانب                 سے                   کی                      391,381
  وجہ                  سے                   کی                      332,634
  نو                   سو                   ہزار                    326,551
  شیئر                 کریں                 کو                      291,827
  ہزار                 نو                   ایک                     276,428
  طرف        

In [12]:
import duckdb
from pathlib import Path

BASE_DIR = Path("checkpoints/merged")

files = {
    "bigram_forward":  BASE_DIR / "bigram_forward.parquet",
    "bigram_backward": BASE_DIR / "bigram_backward.parquet",
    "trigram_left":    BASE_DIR / "trigram_left.parquet",
    "trigram_middle":  BASE_DIR / "trigram_middle.parquet",
    "trigram_right":   BASE_DIR / "trigram_right.parquet",
}

for name, path in files.items():
    total  = duckdb.sql(f"SELECT COUNT(*) FROM read_parquet('{path}')").fetchone()[0]
    kept   = duckdb.sql(f"SELECT COUNT(*) FROM read_parquet('{path}') WHERE cnt >= 5").fetchone()[0]
    unique_keys = duckdb.sql(f"""
        SELECT COUNT(DISTINCT {'(k0,k1)' if 'trigram' in name else 'k0'})
        FROM read_parquet('{path}') WHERE cnt >= 5
    """).fetchone()[0]
    print(f"{name:<20} {total:>12,} → {kept:>12,} rows kept ({kept/total*100:.1f}%)  |  {unique_keys:,} unique keys")

bigram_forward         31,204,242 →    5,266,045 rows kept (16.9%)  |  257,115 unique keys
bigram_backward        31,204,242 →    5,266,045 rows kept (16.9%)  |  244,028 unique keys
trigram_left          126,130,890 →   13,876,178 rows kept (11.0%)  |  2,840,917 unique keys
trigram_middle        126,130,890 →   13,876,178 rows kept (11.0%)  |  5,357,447 unique keys
trigram_right         126,130,890 →   13,876,178 rows kept (11.0%)  |  2,847,884 unique keys


In [ ]:
import joblib
from collections import Counter
from pathlib import Path
from rapidfuzz.distance import Levenshtein

# ── build ─────────────────────────────────────────────────────────────────────
OUTPUT = Path("checkpoints/bk_tree")
OUTPUT.mkdir(parents=True, exist_ok=True)

MILESTONE = 2_000_000
vocab = Counter()

print("Mining vocabulary...")
with open('ur.txt', 'r', encoding='utf-8') as f:
    for line_num, line in enumerate(f, 1):
        vocab.update(normalize_urdu(line).split())
        if line_num % MILESTONE == 0:
            print(f"  {line_num:>10,} lines | vocab: {len(vocab):,}")

print(f"\nTotal unique words: {len(vocab):,}")

Mining vocabulary...
   2,000,000 lines | vocab: 537,242
   4,000,000 lines | vocab: 838,870
   6,000,000 lines | vocab: 1,089,520
   8,000,000 lines | vocab: 1,312,133
  10,000,000 lines | vocab: 1,509,658
  12,000,000 lines | vocab: 1,697,095
  14,000,000 lines | vocab: 1,873,958
  16,000,000 lines | vocab: 2,039,472
  18,000,000 lines | vocab: 2,186,861
  20,000,000 lines | vocab: 2,332,234
  22,000,000 lines | vocab: 2,464,630
  24,000,000 lines | vocab: 2,590,189
  26,000,000 lines | vocab: 2,712,400

Total unique words: 2,827,308

Building BK-Tree...
  0 / 2,827,308 inserted...


TypeError: 'module' object is not callable

In [7]:
# ── BK-Tree ───────────────────────────────────────────────────────────────────
class BKNode:
    __slots__ = ['word', 'count', 'children']
    def __init__(self, word, count):
        self.word = word
        self.count = count
        self.children = {}

class BKTree:
    def __init__(self):
        self.root = None
        self.size = 0

    def insert(self, word, count):
        if self.root is None:
            self.root = BKNode(word, count)
            self.size += 1
            return
        node = self.root
        while True:
            d = Levenshtein.distance(node.word, word)
            if d == 0:
                node.count += count  # merge counts if duplicate
                return
            if d not in node.children:
                node.children[d] = BKNode(word, count)
                self.size += 1
                break
            node = node.children[d]

    def search(self, query, max_dist):
        if self.root is None:
            return []
        results = []
        stack = [self.root]
        while stack:
            node = stack.pop()
            d = Levenshtein.distance(node.word, query)
            if d <= max_dist and d > 0:
                results.append((node.word, node.count, d))
            for dist, child in node.children.items():
                if abs(dist - d) <= max_dist:
                    stack.append(child)
        # sort by distance first, then by count descending
        return sorted(results, key=lambda x: (x[2], -x[1]))

In [ ]:

print("\nBuilding BK-Tree...")
tree = BKTree()
for i, (word, count) in enumerate(vocab.items()):
    tree.insert(word, count)
    if i % 100_000 == 0:
        print(f"  {i:,} / {len(vocab):,} inserted...")

print(f"\nBK-Tree size: {tree.size:,} nodes")


Building BK-Tree...
  0 / 2,827,308 inserted...
  100,000 / 2,827,308 inserted...
  200,000 / 2,827,308 inserted...
  300,000 / 2,827,308 inserted...
  400,000 / 2,827,308 inserted...
  500,000 / 2,827,308 inserted...
  600,000 / 2,827,308 inserted...
  700,000 / 2,827,308 inserted...
  800,000 / 2,827,308 inserted...
  900,000 / 2,827,308 inserted...
  1,000,000 / 2,827,308 inserted...
  1,100,000 / 2,827,308 inserted...
  1,200,000 / 2,827,308 inserted...
  1,300,000 / 2,827,308 inserted...
  1,400,000 / 2,827,308 inserted...
  1,500,000 / 2,827,308 inserted...
  1,600,000 / 2,827,308 inserted...
  1,700,000 / 2,827,308 inserted...
  1,800,000 / 2,827,308 inserted...
  1,900,000 / 2,827,308 inserted...
  2,000,000 / 2,827,308 inserted...
  2,100,000 / 2,827,308 inserted...
  2,200,000 / 2,827,308 inserted...
  2,300,000 / 2,827,308 inserted...
  2,400,000 / 2,827,308 inserted...
  2,500,000 / 2,827,308 inserted...
  2,600,000 / 2,827,308 inserted...
  2,700,000 / 2,827,308 inserted.

In [14]:
for token in list(URDU_0_99.keys()):
    print(URDU_0_99[token])
    tree.insert(URDU_0_99[token],0)
tree.insert('ہزار',0)
tree.insert('لاکھ',0)
tree.insert('سو',0)
tree.insert('کروڑ',0)

صفر
ایک
دو
تین
چار
پانچ
چھ
سات
آٹھ
نو
دس
گیارہ
بارہ
تیرہ
چودہ
پندرہ
سولہ
سترہ
اٹھارہ
انیس
بیس
اکیس
بائیس
تئیس
چوبیس
پچیس
چھبیس
ستائیس
اٹھائیس
انتیس
تیس
اکتیس
بتیس
تینتیس
چونتیس
پینتیس
چھتیس
سینتیس
اڑتیس
انتالیس
چالیس
اکتالیس
بیالیس
تینتالیس
چوالیس
پینتالیس
چھیالیس
سینتالیس
اڑتالیس
انچاس
پچاس
اکیاون
باون
ترپن
چون
پچپن
چھپن
ستاون
اٹھاون
انسٹھ
ساٹھ
اکسٹھ
باسٹھ
ترسٹھ
چونسٹھ
پینسٹھ
چھیاسٹھ
سڑسٹھ
اڑسٹھ
انہتر
ستر
اکہتر
بہتر
تہتر
چوہتر
پچہتر
چھہتر
ستتر
اٹھہتر
اناسی
اسی
اکیاسی
بیاسی
تراسی
چوراسی
پچاسی
چھیاسی
ستاسی
اٹھاسی
نواسی
نوے
اکانوے
بانوے
ترانوے
چورانوے
پچانوے
چھیانوے
ستانوے
اٹھانوے
ننانوے


In [15]:
joblib.dump(tree, OUTPUT / "bk_tree.joblib", compress=3)
print("✅ Done.")

✅ Done.
